# Directorio de Establecimientos de Reclusión del Orden Nacional

Fuente: [datos.gov.co — 24zf-4cfu](https://www.datos.gov.co/Justicia-y-Derecho/Directorio-de-Establecimientos-de-Reclusi-n-de-Ord/24zf-4cfu/about_data) (API Socrata SODA).

In [ ]:
import re
import requests
import pandas as pd

URL = "https://www.datos.gov.co/resource/24zf-4cfu.json"

In [ ]:
def traer_datos(url: str, page_size: int = 1000) -> pd.DataFrame:
    """Descarga todos los registros paginando con $limit/$offset."""
    registros, offset = [], 0
    while True:
        r = requests.get(url, params={"$limit": page_size, "$offset": offset, "$order": ":id"}, timeout=60)
        r.raise_for_status()
        lote = r.json()
        registros.extend(lote)
        if len(lote) < page_size:
            break
        offset += page_size
    return pd.DataFrame(registros)

df_raw = traer_datos(URL)
print(df_raw.shape)
df_raw.head()

## Limpieza
Nombres de columnas legibles y coordenadas convertidas de grados-minutos-segundos (texto con formatos variados) a decimales.

In [ ]:
def dms_a_decimal(valor):
    """Convierte textos como -04°12'35.748\" o 05° 53' 30,96'' a grados decimales."""
    if pd.isna(valor):
        return None
    s = str(valor).strip().replace(",", ".")
    negativo = s.startswith("-") or s.upper().endswith(("S", "W", "O"))
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return None
    grados = nums[0] + (nums[1] / 60 if len(nums) > 1 else 0) + (nums[2] / 3600 if len(nums) > 2 else 0)
    return -grados if negativo else grados

df = df_raw.rename(columns={
    "direcci_n": "direccion",
    "clasificaci_n_art_11_ley": "clasificacion_art_11_ley_65",
    "nombre_establecimiento_sisipec": "nombre_establecimiento",
})
df["latitud"] = df["latitud"].apply(dms_a_decimal).round(6)
df["longitud"] = df["longitud"].apply(dms_a_decimal).round(6)
for c in df.select_dtypes(exclude="number"):
    df[c] = df[c].str.strip()

df.info()
df.head()

In [ ]:
# Coordenadas fuera del rango aproximado de Colombia (revisar manualmente)
fuera = df[~df["latitud"].between(-5, 14) | ~df["longitud"].between(-82, -66)]
fuera[["nombre_establecimiento", "latitud", "longitud"]]

In [ ]:
df.to_csv("establecimientos_reclusion.csv", index=False, encoding="utf-8-sig")